# Sentinel medical faithfulness classifier ? RoBERTa + LoRA on MedHAL-LF

This Colab notebook uses the medical dataset **`GM07/medhal-lf-bal`** and its official `train`, `val`, and `test` splits. It trains only the faithfulness classifier on `(context, statement)` pairs; Sentinel's medical RAG/retrieval and LLM are unchanged. Labels follow the dataset mapping in the supplied workflow: `0 = UNFAITHFUL`, `1 = FAITHFUL`.

In Colab select **Runtime ? Change runtime type ? T4 GPU**. The notebook filters examples with missing/empty context, samples up to 30,000 training and 5,000 validation rows, and uses the complete provided test split. It selects a UNFAITHFUL probability threshold from validation only, then reports held-out recall, accuracy, precision, F1, PR-AUC, ROC-AUC, and risk?coverage/AURC. This compares a trained model; it does not assume RoBERTa improves over DistilBERT.

In [ ]:
%pip -q install transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 datasets==3.1.0 scikit-learn==1.5.2 numpy pandas matplotlib safetensors

In [ ]:
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    auc,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import Dataset

MODEL_NAME = "FacebookAI/roberta-base"
DATASET_NAME = "GM07/medhal-lf-bal"
OUTPUT_DIR = Path("./sentinel_roberta_lora_context")
MAX_LENGTH = 384
TRAIN_SIZE = 30_000
VAL_SIZE = 5_000
SEED = 42
LABEL_TO_ID = {"UNFAITHFUL": 0, "FAITHFUL": 1}
ID_TO_LABEL = {value: key for key, value in LABEL_TO_ID.items()}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
else:
    raise RuntimeError("Enable a Colab GPU runtime (T4 recommended) before training.")
print("Device:", torch.cuda.get_device_name(0))
print("PyTorch CUDA:", torch.version.cuda)

## Load the fixed medical dataset and official splits

Uses the dataset's existing `train`, `val`, and `test` split names, not a random split. The test split is kept intact for final evaluation. No synthetic examples or labels are created.

In [ ]:
dataset = load_dataset(DATASET_NAME)
required_splits = {"train", "val", "test"}
missing_splits = required_splits - set(dataset.keys())
if missing_splits:
    raise ValueError(f"Dataset is missing expected splits: {sorted(missing_splits)}")

required_columns = {"context", "statement", "label"}
for split_name in ("train", "val", "test"):
    missing_columns = required_columns - set(dataset[split_name].column_names)
    if missing_columns:
        raise ValueError(f"{split_name} split is missing columns: {sorted(missing_columns)}")

def has_context(example):
    return example["context"] is not None and str(example["context"]).strip() != ""

train_dataset = dataset["train"].filter(has_context, num_proc=2)
val_dataset = dataset["val"].filter(has_context, num_proc=2)
test_dataset = dataset["test"].filter(has_context, num_proc=2)
print("Context-present train:", len(train_dataset))
print("Context-present val:", len(val_dataset))
print("Context-present test:", len(test_dataset))

train_dataset = train_dataset.shuffle(seed=SEED).select(range(min(TRAIN_SIZE, len(train_dataset))))
val_dataset = val_dataset.shuffle(seed=SEED).select(range(min(VAL_SIZE, len(val_dataset))))

def prepare_labels(example):
    label = int(example["label"])
    if label not in ID_TO_LABEL:
        raise ValueError(f"Expected binary labels 0/1, got {label}")
    return {
        "context": str(example["context"]),
        "statement": str(example["statement"]),
        "labels": label,
    }

train_dataset = train_dataset.map(prepare_labels, remove_columns=train_dataset.column_names, num_proc=2)
val_dataset = val_dataset.map(prepare_labels, remove_columns=val_dataset.column_names, num_proc=2)
test_dataset = test_dataset.map(prepare_labels, remove_columns=test_dataset.column_names, num_proc=2)

for split_name, split in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    counts = np.bincount(np.asarray(split["labels"], dtype=int), minlength=2)
    if np.any(counts == 0):
        raise ValueError(f"{split_name} must contain both labels; counts={counts.tolist()}")
    print(f"{split_name}: {len(split)} examples; UNFAITHFUL={counts[0]}, FAITHFUL={counts[1]}")

## Tokenize context and generated statement as a pair

The model input matches the supplied training workflow: RoBERTa receives `(context, statement)` with a maximum combined length of 384 tokens.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["context"],
        batch["statement"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

tokenized_train = train_dataset.map(
    tokenize_function, batched=True, remove_columns=["context", "statement"], num_proc=2
)
tokenized_val = val_dataset.map(
    tokenize_function, batched=True, remove_columns=["context", "statement"], num_proc=2
)
tokenized_test = test_dataset.map(
    tokenize_function, batched=True, remove_columns=["context", "statement"], num_proc=2
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenized train/validation/test:", len(tokenized_train), len(tokenized_val), len(tokenized_test))

## Configure RoBERTa sequence classification with LoRA

LoRA settings follow the supplied training configuration: rank 8, alpha 16, dropout 0.05, and attention query/key/value targets. The effective training batch is 32 (8 per GPU ? 4 gradient accumulation steps) to fit a free Colab Tesla T4 more reliably.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type="single_label_classification",
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["query", "key", "value"],
    modules_to_save=["classifier"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


def compute_metrics(eval_pred):
    logits = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
    labels = eval_pred.label_ids
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "unfaithful_precision": precision_score(labels, predictions, pos_label=0, zero_division=0),
        "unfaithful_recall": recall_score(labels, predictions, pos_label=0, zero_division=0),
        "unfaithful_f1": f1_score(labels, predictions, pos_label=0, zero_division=0),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
    }

class FaithfulnessTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").long()
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss()(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    learning_rate=3e-5,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    fp16=True,
    tf32=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to="none",
    seed=SEED,
)

trainer = FaithfulnessTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("Training started...")
trainer.train()

## Select the UNFAITHFUL threshold using validation only

Class 0 is UNFAITHFUL. Select its probability threshold by maximizing validation UNFAITHFUL F1 (recall breaks ties). This threshold is locked before evaluation on the held-out test split. We also show the conventional argmax result for comparison.

In [ ]:
import torch.nn.functional as F

def predict_probabilities(dataset):
    output = trainer.predict(dataset)
    logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
    probabilities = F.softmax(torch.as_tensor(logits), dim=-1).cpu().numpy()
    return np.asarray(output.label_ids, dtype=int), probabilities

def metrics_at_threshold(labels, probabilities, threshold):
    labels = np.asarray(labels, dtype=int)
    p_unfaithful = probabilities[:, 0]
    predictions = np.where(p_unfaithful >= threshold, 0, 1)
    return {
        "threshold_unfaithful_probability": float(threshold),
        "accuracy": float(accuracy_score(labels, predictions)),
        "unfaithful_precision": float(precision_score(labels, predictions, pos_label=0, zero_division=0)),
        "unfaithful_recall": float(recall_score(labels, predictions, pos_label=0, zero_division=0)),
        "unfaithful_f1": float(f1_score(labels, predictions, pos_label=0, zero_division=0)),
        "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "confusion_matrix_labels_0_unfaithful_1_faithful": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
    }

val_labels, val_probabilities = predict_probabilities(tokenized_val)
threshold_candidates = np.unique(np.r_[0.0, val_probabilities[:, 0], 1.0])
threshold_results = [metrics_at_threshold(val_labels, val_probabilities, t) for t in threshold_candidates]
best_val = max(threshold_results, key=lambda row: (row["unfaithful_f1"], row["unfaithful_recall"], row["accuracy"]))
UNFAITHFUL_THRESHOLD = best_val["threshold_unfaithful_probability"]
FAITHFUL_ACCEPT_THRESHOLD = 1.0 - UNFAITHFUL_THRESHOLD
print("Validation-selected UNFAITHFUL probability threshold:", UNFAITHFUL_THRESHOLD)
print("Equivalent FAITHFUL acceptance threshold:", FAITHFUL_ACCEPT_THRESHOLD)
print("Validation metrics at selected threshold:", best_val)
print("Validation argmax metrics:", metrics_at_threshold(val_labels, val_probabilities, 0.5))

## Evaluate the test split and calculate risk?coverage/AURC

Use the selected validation threshold once on the test data. Report unfaithful recall (the key safety metric), accuracy, confusion matrix, PR-AUC and ROC-AUC. Risk is the fraction of unfaithful examples among answers accepted, ranked by predicted faithfulness; lower AURC is better.

In [ ]:
test_labels, test_probabilities = predict_probabilities(tokenized_test)
test_report = metrics_at_threshold(test_labels, test_probabilities, UNFAITHFUL_THRESHOLD)
test_report["unfaithful_pr_auc"] = float(average_precision_score(test_labels == 0, test_probabilities[:, 0]))
test_report["unfaithful_roc_auc"] = float(roc_auc_score(test_labels == 0, test_probabilities[:, 0]))
print("Test classification report at selected threshold:")
test_predictions = np.where(test_probabilities[:, 0] >= UNFAITHFUL_THRESHOLD, 0, 1)
print(classification_report(test_labels, test_predictions, labels=[0, 1], target_names=["UNFAITHFUL", "FAITHFUL"], digits=4, zero_division=0))
print("Test confusion matrix (actual rows, predicted columns; labels 0,1):")
print(confusion_matrix(test_labels, test_predictions, labels=[0, 1]))
print("Test metrics:", {key: value for key, value in test_report.items() if "confusion_matrix" not in key})

order = np.argsort(-test_probabilities[:, 1], kind="stable")
ordered_unfaithful = (test_labels[order] == 0).astype(float)
coverage = np.arange(1, len(test_labels) + 1, dtype=float) / len(test_labels)
risk = np.cumsum(ordered_unfaithful) / np.arange(1, len(test_labels) + 1)
aurc = float(auc(np.r_[0.0, coverage], np.r_[0.0, risk]))
test_report["aurc"] = aurc
test_report["risk_coverage"] = {"coverage": coverage.tolist(), "unfaithful_risk": risk.tolist()}
print("Test AURC:", aurc)

import matplotlib.pyplot as plt
plt.plot(coverage, risk)
plt.xlabel("Coverage (fraction of answers accepted)")
plt.ylabel("Unfaithful risk among accepted answers")
plt.title(f"Medical test risk?coverage curve (AURC={aurc:.4f})")
plt.grid(True)
plt.show()

## Save the adapter, tokenizer, and evaluation artifacts

Download the LoRA adapter and metrics bundle from Colab. The acceptance threshold JSON is the threshold Sentinel should use after you integrate the trained adapter; it is derived from validation, not test data.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
report = {
    "dataset": DATASET_NAME,
    "model": MODEL_NAME,
    "labels": ID_TO_LABEL,
    "sizes": {"train": len(tokenized_train), "validation": len(tokenized_val), "test": len(tokenized_test)},
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_macro_f1": trainer.state.best_metric,
    "threshold_selection": "maximize validation UNFAITHFUL F1; recall then accuracy tie-breaks",
    "unfaithful_probability_threshold": UNFAITHFUL_THRESHOLD,
    "faithfulness_accept_threshold": FAITHFUL_ACCEPT_THRESHOLD,
    "validation_at_selected_threshold": best_val,
    "test": test_report,
}
(OUTPUT_DIR / "evaluation.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
(OUTPUT_DIR / "threshold.json").write_text(
    json.dumps({"faithfulness_accept_threshold": FAITHFUL_ACCEPT_THRESHOLD}, indent=2),
    encoding="utf-8",
)
zip_path = Path("./sentinel_roberta_lora_context_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file() and "checkpoint-" not in str(path):
            archive.write(path, path.relative_to(OUTPUT_DIR.parent))
print("Saved bundle:", zip_path.resolve())
from google.colab import files
files.download(str(zip_path))

## Next step: integrate into Sentinel

After verifying the held-out medical results, copy the downloaded adapter and tokenizer into the project's model directory. Load the adapter over `FacebookAI/roberta-base` with PEFT in `src/faithfulness.py`, preserving label IDs `0=UNFAITHFUL`, `1=FAITHFUL`, then read `faithfulness_accept_threshold` from `threshold.json` in the Sentinel decision path. Keep RAG and LLM behavior unchanged. Compare against DistilBERT on the same MedHAL-LF split if you have its actual trained checkpoint; report measured results rather than assuming improvement.